In [1]:
from torchvision import datasets, transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split
import torch
from resnet20 import *


train_transforms = transforms.Compose([
    transforms.Resize((32, 32)),

    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomCrop(32, padding=4),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transforms = transforms.Compose([
    transforms.Resize((32, 32)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


base_dataset = ImageFolder(root=r'C:\github\dataset\archive\train')

train_amount = int(0.8 * len(base_dataset))
val_amount = len(base_dataset) - train_amount

generator = torch.Generator().manual_seed(42)
train_indices, val_indices = random_split(
    range(len(base_dataset)),
    [train_amount, val_amount],
    generator=generator
)

train_folder = ImageFolder(
    root=r'C:\github\dataset\archive\train',
    transform=train_transforms
)

val_folder = ImageFolder(
    root=r'C:\github\dataset\archive\train',
    transform=val_transforms
)

train_set = torch.utils.data.Subset(train_folder, train_indices.indices)
val_set = torch.utils.data.Subset(val_folder, val_indices.indices)

train_loader = DataLoader(
    train_set,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_set,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [2]:
import torch
import torch.nn as nn
import numpy as np
import torchvision

In [7]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = resnet20(2)
model = model.to(device)

In [8]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
epochs = 999

In [9]:
early_stopping_hyperparam = 10
early_stopping_epoch = 0
best_val_loss = None


for e in range(epochs):
    print(f'Epochs {e + 1}:')
    model.train()
    print('TRAINING...')

    total_train_loss = 0
    total_train_acc =0
    train_correct = 0
    num_batches = 0
    for batch in train_loader:
        images, labels = batch
        images, labels = images.to(device), labels.to(device)
        predictions = model(images)
        loss = loss_fn(predictions, labels)
        predicted_classes = torch.argmax(predictions, dim=1)  # FIX: add dim=1
        correct = (predicted_classes == labels).sum().item()
        
        num_batches += 1
        total_train_acc += correct
        total_train_loss += loss.item()
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    print(f'Training Loss: {total_train_loss/num_batches}')
    print(f'Training Acc: {(total_train_acc/(num_batches *32)) * 100:.2f}%')

    model.eval()
    print('VALIDATING...')
    total_val_loss = 0
    total_val_acc =0
    val_correct = 0
    num_batches = 0
    with torch.no_grad():
        for val_batch in val_loader:
            images, labels = val_batch
            images, labels = images.to(device), labels.to(device)
            predictions = model(images)

            predicted_classes = torch.argmax(predictions, dim=1)  # FIX: add dim=1
            correct = (predicted_classes == labels).sum().item()

            num_batches += 1
            total_val_acc += correct
            total_val_loss += loss_fn(predictions, labels).item()


    print(f'Val Loss: {total_val_loss/num_batches}')
    print(f'Val Acc: {(total_val_acc/(num_batches * 32)) * 100:.2f}%')
    if not best_val_loss or total_val_loss < best_val_loss:
        best_val_loss = total_val_loss
        early_stopping_epoch = 0
        print("Saving model")
        torch.save(model.state_dict(), 'model/best_model.pth')
    else:
        early_stopping_epoch += 1
        if early_stopping_epoch >= early_stopping_hyperparam:
            print('Early stopping triggered. Finish Traning.')
            break




Epochs 1:
TRAINING...
Training Loss: 0.27958117588907483
Training Acc: 88.34%
VALIDATING...
Val Loss: 0.20058458394408227
Val Acc: 92.14%
Saving model
Epochs 2:
TRAINING...
Training Loss: 0.21672824212759734
Training Acc: 91.40%
VALIDATING...
Val Loss: 0.16948406996130944
Val Acc: 93.46%
Saving model
Epochs 3:
TRAINING...
Training Loss: 0.19814559209346772
Training Acc: 92.21%
VALIDATING...
Val Loss: 0.18510168448388575
Val Acc: 92.80%
Epochs 4:
TRAINING...
Training Loss: 0.18225775452405216
Training Acc: 92.86%
VALIDATING...
Val Loss: 0.23078426501750945
Val Acc: 90.96%
Epochs 5:
TRAINING...
Training Loss: 0.17063081458732485
Training Acc: 93.36%
VALIDATING...
Val Loss: 0.15092710557729005
Val Acc: 94.27%
Saving model
Epochs 6:
TRAINING...
Training Loss: 0.16003318998739124
Training Acc: 93.73%
VALIDATING...
Val Loss: 0.13686769575327634
Val Acc: 94.59%
Saving model
Epochs 7:
TRAINING...
Training Loss: 0.15425016742758452
Training Acc: 94.14%
VALIDATING...
Val Loss: 0.1583819954633712

KeyboardInterrupt: 